In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter
from PIL import Image 


In [ ]:
class ShotMapPlotter:

    
    def __init__(self, csv_path: str, rink_image_path: str):
        """
        Initialization of the class
        
        Args:
            - csv_path (str): The file path to the cleaned CSV file 
            - rink_image_path (str): The file path to the image of the ice rink
        """
        self.df = pd.read_csv(csv_path, encoding='latin1')
        
        rink_img = Image.open(rink_image_path)
        rink_img_rotated = rink_img.rotate(90, expand=True)
        width, height = rink_img_rotated.size
        rink_half = rink_img_rotated.crop((0, 0, width, height // 2))
        self.rink_image = rink_half
        self.rink_width, self.rink_height = rink_half.size

    
    
    def _normalize_coordinates(self, season_df: pd.DataFrame) -> pd.DataFrame:
        """
        Normalizes shot coordinates for consistent vertical display.

        This method ensures that all shots are represented as if they were heading
        towards the same goal (at the top of the graph). It performs three main operations:
        1. It normalizes all shots so that they are in a single offensive zone (the right half of the rink).
        2. It keeps only shots from the offensive zone.
        3. It swaps the X and Y axes to change from a horizontal orientation to a vertical orientation.

        Args:
            - season_df (pd.DataFrame): A DataFrame containing shot data for a season. 
              Must contain the coordinates and team columns.
            
        Returns:
            - season_df (pd.DataFrame): A DataFrame with the normalized and swapped coordinates 
              using new 'x_norm' and 'y_norm' columns.
        """
        
        season_df['x_temp'] = np.where(season_df['attackingDirection'] == 'left', -season_df['x'], season_df['x'])
        season_df['y_temp'] = np.where(season_df['attackingDirection'] == 'left', -season_df['y'], season_df['y'])
        
        season_df['x_norm'] = season_df['y_temp']
        season_df['y_norm'] = season_df['x_temp']
        
        return season_df
    
    
    def create_interactive_shot_map(self, season_start_year: int, output_html_path: str) -> None:
        """
        Generates and saves an interactive HTML shot map for a given season.

        The shot map is a heat map that shows a team's relative shot rate at a specific location
        compared to the league average. The graph is interactive, allowing the user
        to select a team via a drop-down menu and hover over areas of the rink to get statistics.

        Args:
            - season_start_year (int): The start year of the season to view (e.g., 2018 for 2018-19).
            - output_html_path (str): The file path where to save the HTML graph.
        """

        print(f"Generating the interactive shot map for the {season_start_year}-{season_start_year + 1} season...")

        season_df = self.df[self.df['gameId'].astype(str).str[:4] == str(season_start_year)]
        season_df = self._normalize_coordinates(season_df)

        x_bins = np.linspace(-42.5, 42.5, 44) 
        y_bins = np.linspace(0, 100, 51)

        number_of_games = season_df['gameId'].nunique()
        total_hours = number_of_games * 1

        league_shots_hist, _, _ = np.histogram2d(season_df['x_norm'], season_df['y_norm'], bins=[x_bins, y_bins])
        league_rate_two_teams = league_shots_hist / total_hours
        league_avg_rate_per_hour = league_rate_two_teams / 2

        teams = sorted(season_df['teamName'].dropna().unique())
        team_plot_data = {}

        for team in teams:
            team_df = season_df[season_df['teamName'] == team]
            if team_df.empty: 
                continue

            hours_played_by_team = team_df['gameId'].nunique() * 1
            team_shots_hist, _, _ = np.histogram2d(team_df['x_norm'], team_df['y_norm'], bins=[x_bins, y_bins])
            team_rate_per_hour = team_shots_hist / hours_played_by_team

            relative_rate = team_rate_per_hour - league_avg_rate_per_hour
            
            smoothed_relative = gaussian_filter(relative_rate.T, sigma=2.0)
            smoothed_team = gaussian_filter(team_rate_per_hour.T, sigma=2.0)
            smoothed_league = gaussian_filter(league_avg_rate_per_hour.T, sigma=2.0)

            team_plot_data[team] = {
                'z': smoothed_relative,
                'customdata': np.stack([smoothed_team, smoothed_league], axis=-1)
            }

        global_max_abs_val = 0
        for data in team_plot_data.values():
            max_val = np.nanmax(np.abs(data['z']))
            if max_val > global_max_abs_val:
                global_max_abs_val = max_val
                
        fig = go.Figure()
        first_team = list(team_plot_data.keys())[0]
        fig.add_trace(go.Contour(z=team_plot_data[first_team]['z'],
                                 x=(x_bins[:-1] + x_bins[1:]) / 2,
                                 y=(y_bins[:-1] + y_bins[1:]) / 2,
                                 
                                 zmin=-global_max_abs_val,
                                 zmax=global_max_abs_val,
                                 zmid=0,
                                 colorscale='RdBu_r',
                                 
                                 opacity=0.6,
                                 connectgaps=False,
                                 
                                 customdata=team_plot_data[first_team]['customdata'],
                                 hovertemplate=('<b>Relative Rate: %{z:.3f}</b><br>'
                                                'Team rate: %{customdata[0]:.3f}<br>'
                                                'League average: %{customdata[1]:.3f}<br>'
                                                '<extra></extra>'),
                                 contours=dict(coloring='fill', 
                                               showlabels=False),
                                 colorbar=dict(title='Relative Rate')))
        
        buttons = [dict(method='restyle',
                        label=team,
                        args=[{'z':[data['z']], 'customdata':[data['customdata']]}]) for team, data in team_plot_data.items()]

        rink_aspect_ratio = self.rink_height / self.rink_width 
        rink_width_feet = 42.5 * 2
        rink_height_feet = rink_aspect_ratio * rink_width_feet

        fig.update_layout(title=dict(text=f"Team Shot Map ({season_start_year}-{season_start_year+1} season)",
                                     y=0.9,
                                     x=0.5,
                                     xanchor='center',
                                     yanchor='top'),
                          xaxis=dict(range=[-42.5, 42.5], 
                                     title="Distance from the centre of rink (ft)",
                                     showgrid=True,
                                     gridcolor='rgba(211, 211, 211, 0.2)',
                                     zeroline=False,
                                     dtick=10),
                          yaxis=dict(range=[0, 100], 
                                     title="Distance from the end of the rink (ft)",
                                     scaleanchor="x",
                                     scaleratio=1,
                                     showgrid=True,
                                     gridcolor='rgba(211, 211, 211, 0.2)',
                                     dtick=10),
                          width=650, 
                          height=850,
                          template="plotly_white",
                          margin=dict(t=120),
                          updatemenus=[dict(buttons=buttons, direction="down", showactive=True, x=0.5, xanchor="center", y=1, yanchor="top")],
                          annotations=[],
                          images=[go.layout.Image(source=self.rink_image,
                                                  xref="x", 
                                                  yref="y",
                                                  x=-42.5, 
                                                  y=100,
                                                  sizex=85, 
                                                  sizey=rink_height_feet,
                                                  opacity=1,
                                                  sizing='contain',
                                                  layer="below",
                                                  xanchor="left",
                                                  yanchor="top")])    
                          
        fig.update_yaxes(scaleanchor="x", scaleratio=1)
        fig.update_traces(opacity=0.4)
        fig.write_html(output_html_path)

In [121]:
shotmap = ShotMapPlotter(csv_path='q5_cleaned_data.csv',
                         rink_image_path='../nhl_rink.png')

for year in range(2016, 2024):
    shotmap.create_interactive_shot_map(season_start_year=year, output_html_path=f'{year}-{year+1}.html')

Generating the interactive shot map for the 2016-2017 season...


/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:42: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:43: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Generating the interactive shot map for the 2017-2018 season...


/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:42: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:43: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Generating the interactive shot map for the 2018-2019 season...


...

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:42: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:43: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Generating the interactive shot map for the 2022-2023 season...


/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:42: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:43: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Generating the interactive shot map for the 2023-2024 season...


/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:42: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:43: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/0g/p_g1qhzs21b7wdvfslhc9m9m0000gn/T/ipykernel_81858/3150934599.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th